# FridgeWise AI — Full Data Pipeline (Google Colab)

**Hybrid recipe recommender for food waste reduction**

This notebook runs the complete **Phase 1 data pipeline** for FridgeWise AI:

1. Environment setup
2. Clone project repository
3. Download datasets (Food.com, USDA FDC, FoodKeeper, Open Food Facts)
4. Build ingredient alias dictionary (~1,500 mappings)
5. Clean Food.com recipes & interactions (with expanded normalisation)
6. Enrichment: shelf life, nutrition, synthetic fridges, SQLite database
7. Match-rate analysis (before/after normalisation chart)

**GitHub:** https://github.com/NJDBSProjects20093736/FridgeWise

---

### Before you run
- Enable GPU optional (not required for data pipeline)
- Have a [Kaggle](https://www.kaggle.com) account for Food.com dataset
- Runtime: **GPU or CPU** — data pipeline works on CPU (~5–15 min)

## 1. Install dependencies

In [ ]:
!pip install -q pandas numpy scipy scikit-learn scikit-surprise httpx matplotlib seaborn tqdm python-dateutil kaggle

## 2. Clone repository

In [ ]:
import os
from pathlib import Path

REPO_URL = "https://github.com/NJDBSProjects20093736/FridgeWise.git"
PROJECT_DIR = Path("/content/FridgeWise")

if PROJECT_DIR.exists():
    !rm -rf /content/FridgeWise

!git clone {REPO_URL} /content/FridgeWise
%cd /content/FridgeWise

import sys
sys.path.insert(0, str(PROJECT_DIR))
print("Working directory:", os.getcwd())
!ls -la

## 3. Download Food.com (Kaggle)

Dataset: [food-com-recipes-and-user-interactions](https://www.kaggle.com/datasets/shuyangli94/food-com-recipes-and-user-interactions)

**Option A — Kaggle API (recommended):** Enter your Kaggle username and API key below.

Get your key from: Kaggle → Account → **Create New Token** → downloads `kaggle.json`

In [ ]:
from getpass import getpass
import json
from pathlib import Path

FOOD_COM_DIR = PROJECT_DIR / "data" / "raw" / "food_com"
FOOD_COM_DIR.mkdir(parents=True, exist_ok=True)

recipes_path = FOOD_COM_DIR / "RAW_recipes.csv"
interactions_path = FOOD_COM_DIR / "RAW_interactions.csv"

if recipes_path.exists() and interactions_path.exists():
    print("Food.com files already present — skipping download.")
else:
    kaggle_username = input("Kaggle username: ")
    kaggle_key = getpass("Kaggle API key: ")

    kaggle_dir = Path.home() / ".kaggle"
    kaggle_dir.mkdir(exist_ok=True)
    (kaggle_dir / "kaggle.json").write_text(json.dumps({
        "username": kaggle_username,
        "key": kaggle_key,
    }))
    !chmod 600 ~/.kaggle/kaggle.json

    !kaggle datasets download -d shuyangli94/food-com-recipes-and-user-interactions -p {FOOD_COM_DIR} --unzip
    print("Download complete.")

for f in ["RAW_recipes.csv", "RAW_interactions.csv"]:
    p = FOOD_COM_DIR / f
    print(f"{f}: {'OK' if p.exists() else 'MISSING'} ({p.stat().st_size / 1e6:.1f} MB)" if p.exists() else f"{f}: MISSING")

## 4. Build ingredient alias dictionary

Expands normalisation coverage from the Food.com corpus. Aliases are loaded automatically by the cleaning and enrichment pipelines.

In [ ]:
from src.normalize import load_aliases

!python scripts/build_ingredient_aliases.py

alias_count = load_aliases(PROJECT_DIR / "assets" / "ingredient_aliases.csv")
print(f"Loaded {alias_count} ingredient aliases")

### Option B — Upload Food.com manually (if Kaggle API fails)

Uncomment and run the cell below to upload `RAW_recipes.csv` and `RAW_interactions.csv` from your computer.

In [ ]:
# from google.colab import files
# uploaded = files.upload()  # select RAW_recipes.csv and RAW_interactions.csv
# for name, content in uploaded.items():
#     dest = FOOD_COM_DIR / name
#     dest.write_bytes(content)
#     print(f"Saved {name} -> {dest}")

## 5. Download external datasets

- **USDA FoodData Central** — Foundation Foods CSV (auto)
- **USDA FoodKeeper** — shelf-life JSON (uses fallback if blocked)
- **Open Food Facts** — cached product sample

In [ ]:
!python scripts/download_datasets.py

## 6. Run Food.com cleaning pipeline

Applies 5-core filtering, ingredient normalisation (1,500+ aliases), and saves `clean_recipes.csv` + `clean_interactions.csv`.

In [ ]:
import json
from src.data_pipeline import run_foodcom_pipeline

CLEAN_DIR = PROJECT_DIR / "data" / "clean"
RAW_DIR = PROJECT_DIR / "data" / "raw" / "food_com"

foodcom_stats = run_foodcom_pipeline(RAW_DIR, CLEAN_DIR)
print(json.dumps(foodcom_stats, indent=2))

(CLEAN_DIR / "foodcom_pipeline_stats.json").write_text(
    json.dumps({**foodcom_stats, "filters": {
        "min_user_interactions": 5,
        "min_recipe_interactions": 10,
        "rating_range": "1-5",
        "max_interactions_cap": 100_000,
    }}, indent=2)
)

## 7. Run enrichment pipeline

Builds shelf life, FDC nutrition, Open Food Facts products, context lifts, synthetic fridges, and SQLite database.

In [ ]:
from src.enrichment_pipeline import run_enrichment_pipeline

enrich_stats = run_enrichment_pipeline(PROJECT_DIR)
print(json.dumps(enrich_stats, indent=2))

## 8. Match-rate analysis

Measures fridge ↔ recipe ingredient match before and after normalisation (report Appendix A).

In [ ]:
import json
from src.match_rate import run_match_rate_analysis

match_stats = run_match_rate_analysis(PROJECT_DIR)
print(json.dumps(match_stats, indent=2))

# Display chart inline in Colab
from IPython.display import Image, display
chart_path = PROJECT_DIR / "report" / "appendices" / "match_rate_chart.png"
if chart_path.exists():
    display(Image(filename=str(chart_path)))

## 9. Inspect outputs

In [ ]:
import pandas as pd
import sqlite3

print("=== Clean CSV files ===")
for f in sorted(CLEAN_DIR.glob("*.csv")):
    df = pd.read_csv(f, nrows=0)
    print(f"  {f.name:40s}  columns: {len(df.columns)}")

print("\n=== Sample recipes ===")
recipes = pd.read_csv(CLEAN_DIR / "clean_recipes.csv", nrows=3)
display(recipes[["recipe_id", "recipe_name", "difficulty_level", "n_ingredients"]])

print("\n=== Sample interactions ===")
interactions = pd.read_csv(CLEAN_DIR / "clean_interactions.csv", nrows=3)
display(interactions[["user_id", "recipe_id", "rating", "interaction_date"]])

print("\n=== SQLite tables ===")
db_path = PROJECT_DIR / "data" / "fridge_recommender.db"
conn = sqlite3.connect(db_path)
tables = pd.read_sql("SELECT name FROM sqlite_master WHERE type='table'", conn)
for t in tables["name"]:
    n = pd.read_sql(f"SELECT COUNT(*) AS n FROM [{t}]", conn).iloc[0]["n"]
    print(f"  {t:35s}  {n:>8,} rows")
conn.close()

## 10. Visualise rating distribution

In [ ]:
import matplotlib.pyplot as plt

ratings = pd.read_csv(CLEAN_DIR / "clean_interactions.csv", usecols=["rating"])
ratings["rating"].value_counts().sort_index().plot(kind="bar", color="steelblue", figsize=(8, 4))
plt.title("Rating distribution (clean interactions)")
plt.xlabel("Rating")
plt.ylabel("Count")
plt.tight_layout()
plt.show()

## 11. Save outputs to Google Drive (optional)

Uncomment to copy processed data to your Drive for local use.

In [ ]:
# from google.colab import drive
# drive.mount('/content/drive')
#
# import shutil
# dest = Path("/content/drive/MyDrive/FridgeWise/data")
# dest.mkdir(parents=True, exist_ok=True)
# shutil.copytree(CLEAN_DIR, dest / "clean", dirs_exist_ok=True)
# shutil.copy2(db_path, dest / "fridge_recommender.db")
# print(f"Saved to {dest}")

---

## Phase 2 — Models (coming next)

The following sections will be added as models are implemented:

| Step | Model | Notebook section |
|------|-------|------------------|
| 2.1 | Popularity baseline | §12 |
| 2.2 | Content-based filtering | §13 |
| 2.3 | Collaborative filtering (SVD) | §14 |
| 2.4 | Hybrid recommender | §15 |
| 3.x | Evaluation (MAP, NDCG) | §16 |

---

**Checkpoint 1 complete** when Sections 1–10 run without errors (match-rate uplift ~+2.4 pp).